# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [1]:
import os

# Specify GPU to use (e.g., GPU:0, CPU:-1)
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

# If it fails to determine best cudnn convolution algorithm
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

In [2]:
# Disable all auto-JIT clustering at the process level
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=-1"

### 1.2. Imports

In [3]:
from _imports import * # Centralized file containing all imports

2025-09-10 10:00:41.795205: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-10 10:00:41.812687: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757509241.833867 2199357 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757509241.840390 2199357 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-10 10:00:41.861399: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

### 1.3. GPU Management

In [4]:
get_gpu_info()


TensorFlow GPU Monitor - 2025-09-10 10:00:43
TensorFlow Configuration
Version        : 2.18.0
CUDA Support   : Yes
CUDA Version   : 12.5.1
cuDNN Version  : 9

GPU Information
GPU Name                      Memory Usage         Temp   Util  
--------------------------------------------------------------------------------
0   NVIDIA GeForce RTX 3070      2.3GB /    8.0GB  45C    9%    



2025-09-10 10:00:44.201122: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1757509244.202171 2199357 gpu_device.cc:2022] Created device /device:GPU:0 with 3976 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti, pci bus id: 0000:b3:00.0, compute capability: 8.6


## 2. Run Parameters 

In [5]:
NUM_TRIALS = 2
EPOCHS = 2

SAMPLER_SEED = 111

# Reproducibility settings for TensorFlow:
# Note: must have same inputs and hardware
# Warning: this affects overall performance
# tf.config.experimental.enable_op_determinism() #! Spektral does not support this yet

# Enable or disable XLA compilation
# Note: some layers don't support determinism with XLA
JIT_COMPILE = False  #! Spektral does not support this yet

In [6]:
# Number of top trials to save
TOP_K = 3

# Order to rank trials by:
# "ascending" -> the lowest value is the best
# "descending" -> the highest value is the best
ORDER = "descending"

# Key to rank trials by:
# "value" -> objective trial value
# other e.g., "test_accuracy" -> user params
RANK_KEY = "test_accuracy_s009_full"

# Direction of optimization:
# "minimize" -> the lowest value is the best
# "maximize" -> the highest value is the best
DIRECTION = "minimize"

In [7]:
POLICY = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(POLICY)

BYTES_PER_PARAM = tf.dtypes.as_dtype(POLICY.variable_dtype).size

In [8]:
# Set to an existing dir to resume training
RUN_DIR = f"runs/{get_caller_stem()}"  # (e.g. "runs/nas_1")

## 3. Data Loading and Preprocessing

In [9]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels()

/home/matheus/src/RayWise/src/_load_dataset.py:42: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_train = s008_y_train.astype(np.float32)
/home/matheus/src/RayWise/src/_load_dataset.py:65: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_val = s008_y_val.astype(np.float32)


Shape before conversion: (9234, 8, 32)
Shape after conversion: (9234,)
y_train shape: (9234,)
coord_input shape: (9234, 2)
lidar_input shape: (9234, 20, 200, 10)
Shape before conversion: (1960, 8, 32)
Shape after conversion: (1960,)
y_val shape: (1960,)
coord_input_val shape: (1960, 2)
lidar_input_val shape: (1960, 20, 200, 10)
y_train shape: (11194,)
coord_input shape: (11194, 2)
lidar_input shape: (11194, 20, 200, 10)
Shape before conversion: (9638, 8, 32)
Shape after conversion: (9638,)
y shape: (9638,)
coord_input shape: (9638, 2)
lidar_input shape: (9638, 20, 200, 10)


/home/matheus/src/RayWise/src/_load_dataset.py:101: ComplexWarning: Casting complex values to real discards the imaginary part
  s009_y = s009_y.astype(np.float32)


## 5. Model Definition

In [10]:
def build_model(trial: optuna.Trial, train_seed: int, show_summary: bool = True) -> tf.keras.Model:
    # ———————————————————————————————————————————————————————————————————————————— #
    #                              Model Construction                              #
    # ———————————————————————————————————————————————————————————————————————————— #

    # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
    x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

    # Inline one-hot encoding of semantic values
    one_hot_lidar = layers.Lambda(
        lambda x: tf.concat(
            [
                # “Is there a BS anywhere in the 10 channels?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                # “Vehicle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                # “Obstacle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                # “Free?” → 1 channel (all channels zero)
                tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
            ],
            axis=-1,
        ),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20, 200, 4),
        name="lidar_transform_to_one_hot",
    )(x_lidar_input)
    # -> (batch, 20, 200, 4)

    # Flatten the 20×200 grid into a 4000-length sequence with the 4 channels
    x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, 4), name="lidar_flatten_4_channels")(one_hot_lidar)

    # ———————————————————————————————— GPS Input ———————————————————————————————— #
    # Input for coordinate data (e.g., shape: (2,))
    x_coord_input = layers.Input(shape=(2,), name="coord_input")

    # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
    x_coord: layers.Layer = layers.Lambda(
        lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20 * 200, 2),
        name="coord_tile_flat",
    )(x_coord_input)

    # ————————————————————————————— Combine Branches ————————————————————————————— #
    # Fuse channels:  (batch,4000,4) + (batch,4000,2) → (batch,4000,6)
    combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

    # ———————————————————————————————— Initializer ———————————————————————————————— #
    initializer = tf.keras.initializers.GlorotUniform(
        seed=train_seed,
    )

    # ———————————————————————————————————— CNN ——————————————————————————————————— #
    x1 = layers.Conv1D(
        filters=256,
        kernel_size=5,
        padding="same",
        data_format="channels_last",
        activation=None,
        kernel_initializer=initializer,
        name="conv1d_1",
    )(combined)
    x1 = layers.BatchNormalization(name="conv1d_bn_1")(x1)
    x1 = layers.Activation("sigmoid", name="conv1d_act_1")(x1)
    x1 = layers.MaxPooling1D(pool_size=2, name="max_pool_1")(x1)

    x2 = layers.Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        data_format="channels_last",
        activation=None,
        kernel_initializer=initializer,
        name="conv1d_2",
    )(x1)
    x2 = layers.BatchNormalization(name="conv1d_bn_2")(x2)
    x2 = layers.Activation("relu", name="conv1d_act_2")(x2)
    x2 = layers.MaxPooling1D(pool_size=2, name="max_pool_2")(x2)

    # Align temporal length, keep channels, then concat on the last axis
    _skip12 = resize_for_skip_1d(x1, x2.shape[1], name="skip_cnn1_to_cnn2_resize")
    x2 = layers.Concatenate(axis=-1, name="skip_from_cnn1_to_cnn2")([_skip12, x2])

    x3 = layers.Conv1D(
        filters=128,
        kernel_size=7,
        padding="same",
        data_format="channels_last",
        activation=None,
        kernel_initializer=initializer,
        name="conv1d_3",
    )(x2)
    x3 = layers.BatchNormalization(name="conv1d_bn_3")(x3)
    x3 = layers.Activation("silu", name="conv1d_act_3")(x3)
    x3 = layers.MaxPooling1D(pool_size=3, name="max_pool_3")(x3)

    x4 = layers.Conv1D(
        filters=256,
        kernel_size=7,
        padding="same",
        data_format="channels_last",
        activation=None,
        kernel_initializer=initializer,
        name="conv1d_4",
    )(x3)
    x4 = layers.BatchNormalization(name="conv1d_bn_4")(x4)
    x4 = layers.Activation("gelu", name="conv1d_act_4")(x4)
    x4 = layers.MaxPooling1D(pool_size=3, name="max_pool_4")(x4)

    # ———————————————————————————————————— DNN ——————————————————————————————————— #
    x = x4
    x = layers.Flatten(name="flatten")(x)
    x = layers.Dense(units=500, activation=None, kernel_initializer=initializer, name="dense_1")(x)
    x = layers.Activation("sigmoid", name="dense_act_1")(x)
    x = layers.Dropout(rate=0.2)(x)

    # —————————————————————————————————— Output —————————————————————————————————— #
    outputs = layers.Dense(
        256,
        activation="softmax",
        name="output",
        kernel_initializer=initializer,
    )(x)

    # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

    # ———————————————————————————————— Compilation ——————————————————————————————— #
    model.summary() if show_summary else None

    optimizer = optimizers.Adam(learning_rate=7e-5)

    model.compile(
        optimizer=optimizer,
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=JIT_COMPILE,
    )

    return model

## 6. Objective Function

In [11]:
def objective(
    trial: optuna.Trial,
    *,
    epochs: int = 50,
    size_penalizer: Optional[str] = None,
    **kwargs: Any,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        epochs (int): Number of training epochs.
        size_penalizer (Optional[str]): type of penalizer to use:
            - "params": Penalizes based on the number of parameters.
            - "flops": Penalizes based on the number of FLOPs.
            - None: No penalization is applied.
        **kwargs: Additional keyword arguments.

    Returns:
        float: Final validation loss (optionally penalized) used for optimization.
    """
    (print(f"Running trial {trial.number}..."), clear_session())

    # ————————————————————————————— Seed Search Space ———————————————————————————— #
    DATA_SEED = 356
    TRAIN_SEED = 111

    # ——————————————————————————————————— Setup —————————————————————————————————— #
    # Set Python, NumPy, Keras and TensorFlow seeds
    set_random_seed(TRAIN_SEED)

    global s008_coord_input, s008_lidar_input, s008_y_train
    global s009_coord_input, s009_lidar_input, s009_y

    # ——————————————————————————— New split with Optuna-controlled mix ——————————————————————————— #
    f_s009 = trial.suggest_float("train_frac_s009", 0.0, 0.8)

    split = recreate_train_val_split(f_s009, data_seed=DATA_SEED)

    x_lidar_train = split["x_lidar_train"]
    x_coord_train = split["x_coord_train"]
    y_train = split["y_train"]
    x_lidar_val = split["x_lidar_val"]
    x_coord_val = split["x_coord_val"]
    y_val = split["y_val"]

    # Print train and val shapes
    print("Train labels:", y_train.shape)
    print("Train coords:", x_coord_train.shape)
    print("Train lidar:", x_lidar_train.shape)

    print("Valid labels:", y_val.shape)
    print("Valid coords:", x_coord_val.shape)
    print("Valid lidar:", x_lidar_val.shape)

    backup_dir = kwargs["backup_dir"]
    model_dir = kwargs["model_dir"]
    fig_dir = kwargs["fig_dir"]
    tensorboard_dir = kwargs["tensorboard_dir"]
    logs_dir = kwargs["logs_dir"]
    history_dir = kwargs["history_dir"]

    # ———————————————————————————————————————————————————————————————————————————— #

    try:
        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Data Preprocessing                              #
        # ———————————————————————————————————————————————————————————————————————————— #
        coord_scaler = StandardScaler()

        coord_scaler.fit(x_coord_train)
        x_coord_train = coord_scaler.transform(x_coord_train)
        x_coord_val = coord_scaler.transform(x_coord_val)
        s009_coord_scaled = coord_scaler.transform(s009_coord_input)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                        Model Construction and Training                       #
        # ———————————————————————————————————————————————————————————————————————————— #
        model = build_model(trial=trial, train_seed=TRAIN_SEED, show_summary=True)
        BATCH_SIZE = 64

        prune_model_by_config(
            trial=trial,
            model=model,
            thresholds={
                "model_size": 80,  # Maximum model size in MB
                # "memory_mb": 8000,  # Maximum memory training usage in MB
                # "param": 1e6,  # Maximum number of parameters
                # "flops": 1e9,  # Maximum number of FLOPs
            },
            bytes_per_param=BYTES_PER_PARAM,
            batch_size=BATCH_SIZE,
        )

        history = model.fit(
            x=[x_lidar_train, x_coord_train],
            y=y_train,
            validation_data=([x_lidar_val, x_coord_val], y_val),
            epochs=epochs,
            batch_size=BATCH_SIZE,
            callbacks=get_callbacks_study(
                trial=trial,
                monitor="val_loss",
                #! Can cause high memory usage
                # tensorboard_logs=tensorboard_dir,
                reduce_lr_patience=None,
            ),
            verbose=2,
        )

        trial.set_user_attr("best_train_accuracy", float(max(history.history.get("accuracy", []))))
        trial.set_user_attr("best_val_accuracy", float(max(history.history.get("val_accuracy", []))))

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                            Penalize the Model Size                           #
        # ———————————————————————————————————————————————————————————————————————————— #
        loss_values = punish_model(
            target=history.history["val_loss"],
            model=model,
            type=size_penalizer,
            flops_penalty_factor=1e-10,
            params_penalty_factor=1e-9,
            direction=DIRECTION,
        )
        loss = min(loss_values) if DIRECTION == "minimize" else max(loss_values)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Save Trial Results                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # ——————————————————————————— Model characteristics —————————————————————————— #
        set_user_attr_model_stats(
            trial=trial,
            model=model,
            bytes_per_param=BYTES_PER_PARAM,
            batch_size=BATCH_SIZE,
            n_trials=1000,
            verbose=True,
        )

        # ——————————————————————————— Evaluate on full s009 ——————————————————————————— #
        test_loss_full, test_acc_full = model.evaluate(
            [s009_lidar_input, s009_coord_scaled], s009_y, batch_size=BATCH_SIZE, verbose=0
        )
        trial.set_user_attr("test_accuracy_s009_full", float(test_acc_full))
        trial.set_user_attr("test_loss_s009_full", float(test_loss_full))
        trial.set_user_attr("test_loss_s009_full", float(test_loss_full))

        # ——————————————————————————————— Save history ——————————————————————————————— #
        history_path = os.path.join(history_dir, f"trial_{trial.number}.csv")

        # Create a DataFrame with all history data
        history_data = {
            "epoch": list(range(1, len(history.history["loss"]) + 1)),
            "train_loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
        }

        # Add accuracy metrics if available
        if "accuracy" in history.history:
            history_data["train_accuracy"] = history.history["accuracy"]
        if "val_accuracy" in history.history:
            history_data["val_accuracy"] = history.history["val_accuracy"]

        # Convert to DataFrame and save as CSV
        history_df = pd.DataFrame(history_data)
        history_df.to_csv(history_path, index=False)

        # ————————————————————————— Finish the current trial ————————————————————————— #
        if len(loss_values) > 1:  # Termination Judgement Report
            report_cross_validation_scores(trial, scores=loss_values)

        return loss  # Value to minimize or maximize
    except Exception as e:
        log_trial_error(
            trial=trial,
            exc=e,
            logs_dir=logs_dir,
        )

In [12]:
def recreate_train_val_split(
    train_frac_s009: float,
    *,
    data_seed: int = 356,
) -> Dict[str, np.ndarray]:
    """
    Recreate deterministic train and validation splits using a fraction of s009 data.

    The procedure mirrors the data mixing strategy employed during model training,
    enabling reconstruction of datasets corresponding to any given s009 fraction.
    Notes:
        Produces an exact 80/20 train/validation split over the combined s008 and s009 datasets.
        Only the training split is shuffled; validation ordering remains intact.

    Args:
        train_frac_s009 (float): Fraction of training data sourced from s009.
            Must lie within the inclusive interval [0.0, 0.8].
        data_seed (int): Seed ensuring reproducible splits. Defaults to 356.

    Returns:
        Dict[str, np.ndarray]:
            Mapping containing the arrays: ``x_lidar_train``, ``x_coord_train``,
            ``y_train``, ``x_lidar_val``, ``x_coord_val`` and ``y_val``.

    Raises:
        ValueError: If ``train_frac_s009`` falls outside [0.0, 0.8].
    """
    if not 0.0 <= train_frac_s009 <= 0.8:
        raise ValueError("train_frac_s009 must be between 0.0 and 0.8 inclusive.")

    global s008_coord_input, s008_lidar_input, s008_y_train
    global s009_coord_input, s009_lidar_input, s009_y

    n_s008 = len(s008_y_train)
    n_s009 = len(s009_y)
    total = n_s008 + n_s009

    # Target exact 80/20 split over ALL data
    target_train = int(np.floor(0.8 * total))  # use round(...) if you prefer 16666/4166

    f = train_frac_s009
    rng = np.random.RandomState(data_seed)

    # Reproducible index permutations
    idx_s008 = rng.permutation(n_s008)
    idx_s009 = rng.permutation(n_s009)

    # Choose s009 count closest to f*target_train within feasible bounds
    lower = max(0, target_train - n_s008)  # must leave enough s008 for target_train
    upper = min(n_s009, target_train)  # can’t exceed available s009 or target_train
    n_s009_train = int(np.clip(int(round(f * target_train)), lower, upper))
    n_s008_train = target_train - n_s009_train

    # Train indices
    idx_s009_train = idx_s009[:n_s009_train]
    idx_s008_train = idx_s008[:n_s008_train]
    # Val indices = leftovers (ensures exact 20%)
    idx_s009_val = idx_s009[n_s009_train:]
    idx_s008_val = idx_s008[n_s008_train:]

    # Materialize arrays
    x_s009_lidar_train = s009_lidar_input[idx_s009_train]
    x_s009_coord_train = s009_coord_input[idx_s009_train]
    y_s009_train = s009_y[idx_s009_train]

    x_s008_lidar_train = s008_lidar_input[idx_s008_train]
    x_s008_coord_train = s008_coord_input[idx_s008_train]
    y_s008_train_sel = s008_y_train[idx_s008_train]

    x_s009_lidar_val = s009_lidar_input[idx_s009_val]
    x_s009_coord_val = s009_coord_input[idx_s009_val]
    y_s009_val = s009_y[idx_s009_val]

    x_s008_lidar_val = s008_lidar_input[idx_s008_val]
    x_s008_coord_val = s008_coord_input[idx_s008_val]
    y_s008_val = s008_y_train[idx_s008_val]

    # Merge train/val splits
    x_lidar_train = np.concatenate([x_s009_lidar_train, x_s008_lidar_train], axis=0)
    x_coord_train = np.concatenate([x_s009_coord_train, x_s008_coord_train], axis=0)
    y_train = np.concatenate([y_s009_train, y_s008_train_sel], axis=0)

    x_lidar_val = np.concatenate([x_s009_lidar_val, x_s008_lidar_val], axis=0)
    x_coord_val = np.concatenate([x_s009_coord_val, x_s008_coord_val], axis=0)
    y_val = np.concatenate([y_s009_val, y_s008_val], axis=0)

    # Shuffle TRAIN only
    perm = rng.permutation(len(y_train))
    x_lidar_train = x_lidar_train[perm]
    x_coord_train = x_coord_train[perm]
    y_train = y_train[perm]

    return {
        "x_lidar_train": x_lidar_train,
        "x_coord_train": x_coord_train,
        "y_train": y_train,
        "x_lidar_val": x_lidar_val,
        "x_coord_val": x_coord_val,
        "y_val": y_val,
    }

## Main

In [ ]:
study = run_study(
    objective=objective,
    run_dir=RUN_DIR,
    epochs=EPOCHS,
    num_trials=NUM_TRIALS,
    sampler_seed=SAMPLER_SEED,
    direction=DIRECTION,
    top_k=TOP_K,
    rank_key=RANK_KEY,
    order=ORDER,
    extra_attrs=[
        "best_train_accuracy",
        "best_val_accuracy",
    ],
    variance_threshold=None,
    prune_threshold=None,
    patience=None,
)

[I 2025-09-10 10:00:46,353] A new study created in RDB with name: optuna_study


Running trial 0...
Train labels: (15743,)
Train coords: (15743, 2)
Train lidar: (15743, 20, 200, 10)
Valid labels: (3161,)
Valid coords: (3161, 2)
Valid lidar: (3161, 20, 200, 10)


I0000 00:00:1757509247.095282 2199357 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3976 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti, pci bus id: 0000:b3:00.0, compute capability: 8.6


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ lidar_input         │ (None, 20, 200,   │          0 │ -                 │
│ (InputLayer)        │ 10)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lidar_transform_to… │ (None, 20, 200,   │          0 │ lidar_input[0][0] │
│ (Lambda)            │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ coord_input         │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lidar_flatten_4_ch… │ (None, 4000, 4)   │          0 │ lidar_transform_… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ coord_tile_flat     │ (None, 4000, 2)   │          0 │ coord_input[0][0] │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combine_lidar_coord │ (None, 4000, 6)   │          0 │ lidar_flatten_4_… │
│ (Concatenate)       │                   │            │ coord_tile_flat[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 4000, 256) │      7,936 │ combine_lidar_co… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_bn_1         │ (None, 4000, 256) │      1,024 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_act_1        │ (None, 4000, 256) │          0 │ conv1d_bn_1[0][0] │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pool_1          │ (None, 2000, 256) │          0 │ conv1d_act_1[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 2000, 64)  │     49,216 │ max_pool_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_bn_2         │ (None, 2000, 64)  │        256 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_act_2        │ (None, 2000, 64)  │          0 │ conv1d_bn_2[0][0] │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ skip_cnn1_to_cnn2_… │ (None, 1000, 256) │          0 │ max_pool_1[0][0]  │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pool_2          │ (None, 1000, 64)  │          0 │ conv1d_act_2[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ skip_from_cnn1_to_… │ (None, 1000, 320) │          0 │ skip_cnn1_to_cnn… │
│ (Concatenate)       │                   │            │ max_pool_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 1000, 128) │    286,848 │ skip_from_cnn1_t… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_bn_3         │ (None, 1000, 128) │        512 │ conv1d_3[0][0]  

...

 Total params: 14,913,204 (56.89 MB)

 Trainable params: 14,911,796 (56.88 MB)

 Non-trainable params: 1,408 (5.50 KB)

2025-09-10 10:02:04.442607: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: INVALID_ARGUMENT: Expected multiples argument to be a vector of length 4 but got length 3
/home/matheus/anaconda3/envs/tf-optuna-araras/lib/python3.11/site-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ('lidar_input', 'coord_input'). Received: the structure of inputs=['*', '*']
  warnings.warn(


Epoch 1/2


In [ ]:
best_frac = float(study.best_trial.params["train_frac_s009"])
split = recreate_train_val_split(best_frac)

output_dir = Path("best_datasets")
train_dir = output_dir / "train"
valid_dir = output_dir / "valid"
train_dir.mkdir(parents=True, exist_ok=True)
valid_dir.mkdir(parents=True, exist_ok=True)

np.save(train_dir / "labels.npy", split["y_train"])
np.save(train_dir / "coords.npy", split["x_coord_train"])
np.save(train_dir / "lidar.npy", split["x_lidar_train"])

np.save(valid_dir / "labels.npy", split["y_val"])
np.save(valid_dir / "coords.npy", split["x_coord_val"])
np.save(valid_dir / "lidar.npy", split["x_lidar_val"])

In [ ]:
# Load best datasets
output_dir = Path("best_datasets")
train_dir = output_dir / "train"
valid_dir = output_dir / "valid"

# Load train splits
y_train = np.load(train_dir / "labels.npy")
x_coord_train = np.load(train_dir / "coords.npy")
x_lidar_train = np.load(train_dir / "lidar.npy")

# Load validation splits
y_val = np.load(valid_dir / "labels.npy")
x_coord_val = np.load(valid_dir / "coords.npy")
x_lidar_val = np.load(valid_dir / "lidar.npy")

# Print shapes
print("Train labels:", y_train.shape)
print("Train coords:", x_coord_train.shape)
print("Train lidar:", x_lidar_train.shape)

print("Valid labels:", y_val.shape)
print("Valid coords:", x_coord_val.shape)
print("Valid lidar:", x_lidar_val.shape)

Train labels: (16665,)
Train coords: (16665, 2)
Train lidar: (16665, 20, 200, 10)
Valid labels: (4167,)
Valid coords: (4167, 2)
Valid lidar: (4167, 20, 200, 10)
